# Building an MCP Interface for Internet Topology Data

This is the single student notebook deliverable. Run it from top to bottom after completing the MCP tools. Every dataset access in this notebook goes through the authenticated MCP client; do not add database drivers, connection strings, or SQL. Preserve each call's evidence record with your submitted notebook.

Before starting Jupyter, export the variables from `itdk_mcp_credentials.env` and start the local services with `docker compose --env-file itdk_mcp_credentials.env up -d`.

## Complete setup (do not edit)

These imports and helpers open short authenticated MCP sessions, capture tool arguments and structured result metadata, and read only CSV files produced by the MCP server. The bearer key is never displayed.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import pandas as pd
from mcp import ClientSession
from mcp.client.sse import sse_client

MCP_URL = os.environ.get("ITDK_MCP_URL", "http://127.0.0.1:8000/mcp/sse")
SNAPSHOT_ID = os.environ.get("ITDK_SNAPSHOT_ID", "nids-itdk-mcp-synthetic-v1")
SERVER_OUTPUT_DIR = Path(os.environ.get("ITDK_SERVER_OUTPUT_DIR", "/app/outputs"))
LOCAL_OUTPUT_DIR = Path(os.environ.get("ITDK_OUTPUT_DIR", "./outputs"))
MASTER_KEY = os.environ.get("MCP_MASTER_KEY", "")
if not MASTER_KEY:
    raise RuntimeError("Export MCP_MASTER_KEY before starting Jupyter; do not paste it into this notebook.")

async def _session_call(action: str, name: str | None = None, arguments: dict[str, Any] | None = None) -> Any:
    headers = {"Authorization": f"Bearer {MASTER_KEY}"}
    async with sse_client(MCP_URL, headers=headers) as streams:
        async with ClientSession(*streams) as session:
            await session.initialize()
            if action == "list":
                return await session.list_tools()
            assert name is not None and arguments is not None
            return await session.call_tool(name, arguments)

async def list_itdk_tools() -> list[dict[str, Any]]:
    result = await _session_call("list")
    return [tool.model_dump(mode="json", by_alias=True) for tool in result.tools]

async def call_itdk_raw(name: str, arguments: dict[str, Any]) -> dict[str, Any]:
    result = await _session_call("call", name, arguments)
    return {"snapshot": SNAPSHOT_ID, "tool_name": name, "arguments": dict(arguments), "result": result.model_dump(mode="json", by_alias=True)}

async def call_itdk_tool(name: str, arguments: dict[str, Any]) -> dict[str, Any]:
    raw_evidence = await call_itdk_raw(name, arguments)
    payload = raw_evidence["result"]
    if payload.get("isError"):
        raise RuntimeError(f"{name} returned an MCP error: {payload.get('content')!r}")
    metadata = payload.get("structuredContent")
    if not isinstance(metadata, dict):
        raise RuntimeError(f"{name} returned no structuredContent metadata")
    return {"snapshot": SNAPSHOT_ID, "tool_name": name, "arguments": dict(arguments), "metadata": metadata}

def local_csv_path(evidence: dict[str, Any]) -> Path:
    server_path = Path(evidence["metadata"]["file_path"])
    relative_path = server_path.relative_to(SERVER_OUTPUT_DIR)
    return LOCAL_OUTPUT_DIR / relative_path

def load_tool_csv(evidence: dict[str, Any]) -> pd.DataFrame:
    frame = pd.read_csv(local_csv_path(evidence), keep_default_na=False, na_values=[r"\N"])
    expected = int(evidence["metadata"]["row_count"])
    if len(frame) != expected:
        raise AssertionError(f"CSV row count {len(frame)} does not match metadata {expected}")
    return frame

def show_evidence(evidence: dict[str, Any]) -> None:
    print(json.dumps(evidence, indent=2, sort_keys=True))

## Task 0 — Environment and protocol orientation

Run this complete cell before editing server TODOs. It records discovery and a successful call to the worked example; it does not connect to PostgreSQL directly.

In [ ]:
discovered_tools = await list_itdk_tools()
orientation_tool_names = [tool["name"] for tool in discovered_tools]
orientation_l1_evidence = await call_itdk_tool("get_link_endpoints", {"link_id": "L1"})
print("Discovered tools:", orientation_tool_names)
print(json.dumps(next(tool for tool in discovered_tools if tool["name"] == "get_link_endpoints"), indent=2, sort_keys=True))
show_evidence(orientation_l1_evidence)

## Task 1 — Understand the ITDK representation

### Task 1.1 — Link and Node Identifiers

In [ ]:
# YOUR CODE HERE
# Output: link_l1_df, a (2, 4) DataFrame ordered by endpoint_ordinal.
# Hint: call get_link_endpoints with the documented synthetic link L1.
link_l1_evidence = None
link_l1_df = None

**Q1** For the supplied fixture link, explain the different roles of `link_id`, `endpoint_ordinal`, `endpoint_token`, and `node_id`, citing concrete returned rows.

*Your answer for Q1:*

Replace this line with your answer.

### Task 1.1 — Link and Node Identifiers

In [ ]:
# YOUR CODE HERE
# Output: endpoint_key_evidence, a DataFrame showing endpoint_token and node_id for L1.
# Hint: select those two columns from link_l1_df; do not split endpoint_token to create a join key.
endpoint_key_evidence = None

**Q2** Which field is the normal key for relating an endpoint to AS or geolocation rows, and what goes wrong if `endpoint_token` is used as though it were that key?

*Your answer for Q2:*

Replace this line with your answer.

### Task 1.2 — Multiplicity and Provenance

In [ ]:
# YOUR CODE HERE
# Output: n2_asn_df with two N2 ASN rows and n2_links_df with N2's link rows.
# Hint: combine find_nodes_by_asn calls for 64500 and 64501, then call find_links_for_node for N2.
n2_asn_evidence = []
n2_asn_df = None
n2_links_evidence = None
n2_links_df = None

**Q3** Use the supplied fixture nodes to show two different one-to-many patterns in the four-relation model. Why would flattening all annotations into one assumed row per router lose information or multiply rows?

*Your answer for Q3:*

Replace this line with your answer.

### Task 1.2 — Multiplicity and Provenance

In [ ]:
# YOUR CODE HERE
# Output: asn_method_counts and geo_method_counts, each indexed by method.
# Hint: call find_nodes_by_asn for 64500/64501 and search_nodes_by_geolocation for US/DE.
asn_method_counts = None
geo_method_counts = None

**Q4** What do the returned AS-assignment and geolocation `method` fields communicate? Explain why those values should remain in an evidence table.

*Your answer for Q4:*

Replace this line with your answer.

### Task 1.2 — Multiplicity and Provenance

In [ ]:
# YOUR CODE HERE
# Output: missing_ptr_df, an empty (0, 2) DataFrame for the documented N6 interface IP.
# Hint: obtain 203.0.113.6 from get_link_endpoints(L5), then call lookup_router_hostnames by IP.
link_l5_evidence = None
missing_ptr_evidence = None
missing_ptr_df = None

**Q5** Identify one missing PTR, location, interface encoding, or annotation in the fixture. What can you conclude from the missing value, and what tempting conclusion is not justified?

*Your answer for Q5:*

Replace this line with your answer.

## Task 2 — Implement constrained lookup tools

### Task 2.1 — Find Nodes by ASN

In [ ]:
# YOUR CODE HERE
# Output: one successful evidence record, two raw records whose result.isError is true, and the name of a no-dispatch test.
# Hint: try integer 64500, string '64500', and integer 64500 plus an extra property.
valid_asn_evidence = None
invalid_string_asn_result = None
invalid_extra_asn_result = None
asn_no_dispatch_test = "YOUR TEST NAME HERE"

**Q6** Show one valid and two meaningfully different invalid calls to `find_nodes_by_asn`. Explain which contract rule handles each invalid call and prove that invalid inputs do not reach the repository.

*Your answer for Q6:*

Replace this line with your answer.

### Task 2.2 — Search Nodes by Geolocation

In [ ]:
# YOUR CODE HERE
# Output: geolocation_contract plus the name and result of your country-first query test.
# Hint: obtain the advertised schema through list_itdk_tools; cite repository test evidence without adding SQL here.
geolocation_contract = None
country_first_test = "YOUR TEST NAME AND RESULT HERE"

**Q7** Why does this tool require `country` even when coordinate bounds are present, and how does your query preserve the intended indexed access path?

*Your answer for Q7:*

Replace this line with your answer.

### Task 2.2 — Search Nodes by Geolocation

In [ ]:
# YOUR CODE HERE
# Output: a successful bounded result, a successful empty result, and an inverted-bound record whose result.isError is true.
# Hint: use US with [-125,-120] x [45,50], US with [-10,-5] x [0,5], then longitude_min=10 and longitude_max=-10.
bounded_geo_evidence = None
bounded_geo_df = None
empty_geo_evidence = None
empty_geo_df = None
inverted_geo_result = None

**Q8** Demonstrate a valid bounded search and an invalid inverted-bound search. Record the exact arguments and explain how your tests distinguish an empty valid result from `INVALID_ARGUMENT`.

*Your answer for Q8:*

Replace this line with your answer.

## Task 3 — Implement mutually exclusive hostname selectors

### Task 3.1 — Define and Enforce Exactly One Selector

In [ ]:
# YOUR CODE HERE
# Output: a successful one-selector record plus raw zero/two-selector records whose result.isError is true.
# Hint: use ip=192.0.2.1 for the successful call, then {} and both ip/hostname_exact.
one_selector_evidence = None
zero_selector_result = None
two_selector_result = None

**Q9** Why are the three selectors mutually exclusive? Demonstrate the behavior for zero, one, and two supplied selectors and identify which layer or layers preserve the invariant.

*Your answer for Q9:*

Replace this line with your answer.

### Task 3.2 — Treat Prefix Metacharacters Literally

In [ ]:
# YOUR CODE HERE
# Output: literal_prefix_frames for edge%, edge_, and edge\, each containing only its literal fixture match.
# Hint: call hostname_prefix separately for all three metacharacter prefixes and preserve every evidence record.
literal_prefix_evidence = {}
literal_prefix_frames = {}

**Q10** Using the fixture's metacharacter cases, show that `%`, `_`, and backslash in a requested prefix are ordinary characters. What incorrect rows would a non-literal implementation have matched?

*Your answer for Q10:*

Replace this line with your answer.

## Task 4 — Add one complete MCP capability

### Task 4.1 — Wire the Vertical Slice

In [ ]:
# YOUR CODE HERE
# Output: find_links_contract, one evidence record/DataFrame, and a layer_trace table with one row per layer.
# Hint: discover and call find_links_for_node for N2, then relate schema, dispatch, repository and CSV metadata evidence.
find_links_contract = None
find_links_trace_evidence = None
find_links_trace_df = None
layer_trace = None

**Q11** Trace one `find_links_for_node` call from discovery through CSV publication. What does each layer contribute, and how did you verify that all layers agree on the contract?

*Your answer for Q11:*

Replace this line with your answer.

### Task 4.2 — Test Through MCP

In [ ]:
# YOUR CODE HERE
# Output: student_test_summary naming your most useful test, its layer, observed result, and protected failure.
# Hint: cite one student-authored test and compare it with a test at a different contract layer.
student_test_summary = None

**Q12** Describe the most useful test you added for this tool, the failure it would catch, and why a test at a different layer would not prove the same behavior.

*Your answer for Q12:*

Replace this line with your answer.

## Task 5 — Use the completed MCP to investigate topology

### Task 5.1 — Start From an ASN

In [ ]:
# YOUR CODE HERE
# Output: selected_asn_summary with ASN, row count, unique-node count, and assignment-method counts.
# Hint: use the documented seed ASN 64500 and preserve its evidence record.
selected_asn_evidence = None
selected_asn_df = None
selected_asn_summary = None

**Q13** For the instructor-provided ASN, how many distinct router nodes are returned and which assignment methods occur? Explain why the result is a snapshot-specific assignment count rather than a complete current inventory of the AS.

*Your answer for Q13:*

Replace this line with your answer.

### Task 5.2 — Follow Links From Two Nodes

In [ ]:
# YOUR CODE HERE
# Output: neighbor_evidence_df with every endpoint for every link containing N1 or N2.
# Hint: reproducibly choose the first two distinct node IDs from the stable ASN result, find their links, deduplicate link_id, then call get_link_endpoints for each link.
selected_node_link_evidence = []
endpoint_evidence = []
neighbor_evidence_df = None

**Q14** Choose two returned nodes using a reproducible rule. For each, what links contain it, what endpoint rows represent its immediate router-level neighbors, and which records require special care because a link has other than two endpoint rows or an endpoint lacks an interface encoding?

*Your answer for Q14:*

Replace this line with your answer.

### Task 5.3 — Compare Geographic Result Sets

In [ ]:
# YOUR CODE HERE
# Output: country_comparison with one row per country and country_plot with labeled grain and units.
# Hint: call search_nodes_by_geolocation for US and DE, apply the same summary, and use a categorical count plot.
country_evidence = {}
country_frames = {}
country_comparison = None
country_plot = None

**Q15** Compare the instructor-provided two countries or bounded regions with a clearly defined summary and visualization. What differences are visible, and which broader geographic conclusions would be unjustified from these results alone?

*Your answer for Q15:*

Replace this line with your answer.

### Task 5.4 — Assemble and Audit Evidence

In [ ]:
# YOUR CODE HERE
# Output: topology_evidence_table joining names, nodes, links, ASN rows, and locations where supported; agent_claim_audit listing each factual claim and its evidence status.
# Hint 1: start with IP 192.0.2.1, match it only to eligible interface evidence already returned by MCP, then work outward with approved calls.
# Hint 2: merge only on documented keys; keep call records in topology_call_evidence.
# Hint 3: paste a short agent proposal below, split it into factual claims, and mark supported/unsupported/overstated with exact evidence references.
topology_call_evidence = []
topology_evidence_table = None
agent_proposal = """PASTE A SHORT MCP-CAPABLE AGENT PROPOSAL HERE"""
agent_claim_audit = None

**Q16** Starting from the supplied IP or hostname seed, build a reproducible evidence table connecting names, interfaces, nodes, links, AS assignments, and locations wherever the available tools permit it. Then ask the approved MCP-capable agent to propose a short interpretation, verify every factual claim against your captured CSVs, and identify at least one claim that is overstated, unsupported, or missing an essential limitation. If no course agent is available, use the instructor-provided deterministic fallback prompt/output.

*Your answer for Q16:*

Replace this line with your answer.

Before submission, restart the kernel and run all cells. Confirm there are no direct database imports or SQL strings and that every answer cites tool arguments, artifact metadata, row counts, transformations, and limitations.